# 00 · Setup and validation
Builds the tools and checks that this Kaggle machine reproduces the project's simulation setup before anything else. Runs two 77 K/100 bar GCMC simulations of top MOFs and compares them with the original values (OJOSEN 62.5 g/L, PIDNEX 55.8 g/L). **Runtime: about 30 minutes.**

**Attach:** the `mofh2_pipeline` dataset.

Force field (identical everywhere): UFF framework; single-site H₂ (ε/k = 36.7 K, σ = 2.958 Å); Feynman–Hibbs correction with mass 2.016 u for H₂–H₂ (hMOF-matched); no charges; 12.8 Å cutoff.

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = True
MAX_HOURS = 11.0
STAGE_QUEUES = []
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Validation runs

In [ ]:
%cd $WORK
open('queues/00_validation.txt','w').write(
  'C19_OJOSEN_clean|77|100|400|UFF_FHm2|2 2 2|runs_val/C19_OJOSEN_clean_100\n'
  'C19_PIDNEX_clean|77|100|400|UFF_FHm2|1 1 2|runs_val/C19_PIDNEX_clean_100\n')
import pandas as pd
ref = pd.read_csv('screen_top20_ft_gcmc.csv').set_index('raspa')
lines = [l for l in open('queues/00_validation.txt') if l.strip()]
lines = [l.split('|') for l in lines]
for l in lines: l[5] = ref.loc[l[0], 'cells']
open('queues/00_validation.txt','w').write(''.join('|'.join(l) for l in lines))
!cat queues/00_validation.txt
!bash rq_any.sh queues/00_validation.txt 2

In [ ]:
import sys; sys.path.insert(0, WORK)
from extract import parse_data
for n, exp in [('C19_OJOSEN_clean', 62.5), ('C19_PIDNEX_clean', 55.8)]:
    f = glob.glob(f'{WORK}/runs_val/{n}_100/Output/System_0/*.data')[0]; v = parse_data(f)['gL']
    print(f'{n}: {v:.2f} g/L vs original {exp} -> {"PASS" if abs(v-exp) < 2 else "FAIL"}')

### Model checkpoints load and predict
The 4-seed graph-network ensemble trained on hMOF (from the project's GitHub repository). Expected: MOF-5 (IRMOF-1) about 52.9 g/L at 77 K/100 bar.

In [ ]:
import torch, numpy as np, predict_topo as P
from mofnet import MOFNet
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
f = P._featurise(f'{WORK}/raspa/share/raspa/structures/cif/IRMOF-1.cif')
g = Data(x=torch.tensor(f['x'],dtype=torch.float), edge_index=torch.tensor(f['edges']).t().contiguous(), edge_attr=torch.tensor(f['dists'],dtype=torch.float))
g.gfeat = torch.tensor([[float(f[c]) for c in P.GLOBALS]], dtype=torch.float)
preds = []
for c in sorted(glob.glob(P.REPO+'/models/gnn_full_random_s*.pth')+glob.glob(P.REPO+'/results/gnn_full_random_s*.pth')):
    m = MOFNet(); m.load_state_dict(torch.load(c, map_location='cpu', weights_only=True)); m.eval()
    with torch.no_grad(): preds.append((m(next(iter(DataLoader([g]))))*m.y_sd+m.y_mu).numpy()[0])
print('IRMOF-1 ensemble prediction (2 bar, 100 bar, DC) g/L:', np.mean(preds,0).round(2))